# Notebook 1: Pipeline Parity — R vs Python

比较 py-SpaceTrooper (Python) 与 SpaceTrooper (R) 的完整 QC 管道输出。

## 管道步骤
1. 读取 CosMx 数据
2. `spatialPerCellQC` — 计算每细胞 QC 指标
3. `computeOutliersQCScore` — 计算异常值
4. `checkOutliers` — 验证异常值数量
5. `computeQCScore` — 岭逻辑回归 QC 评分

## Parity 门控
- QC_score: Pearson r ≥ 0.90
- 中间指标: max abs error < 1e-8

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import anndata as ad

# 加载 R 参考输出
with open('../data/r_reference_output.json') as f:
    r_ref = json.load(f)

print(f'R reference: {r_ref["n_cells"]} cells, {r_ref["n_genes"]} genes')
print(f'R formula variables: {r_ref["formula_variables"]}')

In [ ]:
# 运行 Python 管道
np.random.seed(42)
adata = ad.read_h5ad('../data/fixture_cosmx.h5ad')

from spacetrooper.qc_metrics import spatial_per_cell_qc
from spacetrooper.qc_score import compute_qc_score

spatial_per_cell_qc(adata)
compute_qc_score(adata, verbose=False)

print(f'Python: {adata.n_obs} cells, {adata.n_vars} genes')
print(f'Python formula variables: {list(adata.uns["formula_variables"].values())}')

In [ ]:
# 对齐并比较 QC_score
r_ids = r_ref['cell_id']
py_ids = adata.obs.index.tolist()
common = sorted(set(r_ids).intersection(set(py_ids)))
r_idx = [r_ids.index(c) for c in common]
py_idx = [py_ids.index(c) for c in common]

r_scores = np.array([r_ref['QC_score'][i] for i in r_idx])
py_scores = adata.obs['QC_score'].values[py_idx]

corr = np.corrcoef(r_scores, py_scores)[0, 1]
max_err = np.max(np.abs(r_scores - py_scores))
mean_err = np.mean(np.abs(r_scores - py_scores))

print(f'=== QC_score Parity ===')
print(f'Pearson r:      {corr:.4f}')
print(f'Max abs error:  {max_err:.4e}')
print(f'Mean abs error: {mean_err:.4e}')
print(f'Gate (r >= 0.90): {"PASS" if corr >= 0.90 else "FAIL"}')

In [ ]:
# 散点图: R vs Python QC_score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(r_scores, py_scores, s=5, alpha=0.5)
ax.plot([0, 1], [0, 1], 'r--', linewidth=1)
ax.set_xlabel('R QC_score')
ax.set_ylabel('Python QC_score')
ax.set_title(f'R vs Python QC_score (Pearson r = {corr:.4f})')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax = axes[1]
ax.hist(r_scores, bins=30, alpha=0.5, label='R')
ax.hist(py_scores, bins=30, alpha=0.5, label='Python')
ax.set_xlabel('QC_score')
ax.set_ylabel('Count')
ax.set_title('QC_score Distribution')
ax.legend()

plt.tight_layout()
plt.savefig('compare_R_vs_Python.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 中间指标比较
metrics = ['log2SignalDensity', 'Area_um', 'ctrl_total_ratio']

fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 4))

for i, metric in enumerate(metrics):
    if metric in r_ref:
        r_vals = np.array([r_ref[metric][j] for j in r_idx])
        py_vals = adata.obs[metric].values[py_idx]
        mask = ~np.isnan(r_vals) & ~np.isnan(py_vals)
        if mask.sum() > 0:
            err = np.max(np.abs(r_vals[mask] - py_vals[mask]))
            axes[i].scatter(r_vals[mask], py_vals[mask], s=3, alpha=0.3)
            axes[i].set_xlabel(f'R {metric}')
            axes[i].set_ylabel(f'Python {metric}')
            axes[i].set_title(f'{metric}\nmax_err = {err:.2e}')

plt.tight_layout()
plt.show()